In [3]:
import os

# Workaround to avoid Windows DLL initialization errors for some GPU/cuda setups:
# - disable CUDA device visibility so torch doesn't try to initialize CUDA drivers
# - allow duplicate OpenMP libs if needed (sometimes helps with DLL init issues)
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

try:
	import torch
	import torch.nn as nn
	from torch.utils.data import Dataset, DataLoader
except OSError as e:
	# Provide actionable message; re-raise so notebook stops and user can take action
	print("Failed to import torch due to OSError:", e)
	print("Possible fixes:")
	print(" - Install/repair Microsoft Visual C++ Redistributable (2015-2019 / 2015-2022).")
	print(" - Reinstall a CPU-only PyTorch wheel to avoid GPU/CUDA DLL issues.")
	print("   Example pip command (run in the notebook or terminal):")
	print("     %pip install --upgrade --force-reinstall --index-url https://download.pytorch.org/whl/cpu torch")
	raise

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix, classification_report
import re
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

Failed to import torch due to OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\iryna.sitka\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.
Possible fixes:
 - Install/repair Microsoft Visual C++ Redistributable (2015-2019 / 2015-2022).
 - Reinstall a CPU-only PyTorch wheel to avoid GPU/CUDA DLL issues.
   Example pip command (run in the notebook or terminal):
     %pip install --upgrade --force-reinstall --index-url https://download.pytorch.org/whl/cpu torch


OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\iryna.sitka\AppData\Local\Programs\Python\Python313\Lib\site-packages\torch\lib\c10.dll" or one of its dependencies.

In [ ]:
true_df = pd.read_csv('lab6/True.csv')
fake_df = pd.read_csv('lab6/Fake.csv')

true_df['label'] = 0  
fake_df['label'] = 1  
df = pd.concat([true_df, fake_df], ignore_index=True)
df['text'] = df['title'] + " " + df['text']
df = df[['text', 'label']]

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    return text.strip()

df['text'] = df['text'].apply(clean_text)


In [ ]:
all_words = ' '.join(df['text']).split()
vocab = ['<PAD>', '<UNK>'] + [word for word, count in Counter(all_words).most_common(10000)]
word_to_idx = {word: i for i, word in enumerate(vocab)}

In [ ]:
def text_to_indices(text, max_len=200):
    words = text.split()
    indices = [word_to_idx.get(word, 1) for word in words[:max_len]]
    if len(indices) < max_len:
        indices += [0] * (max_len - len(indices))
    return indices[:max_len]

df['indices'] = df['text'].apply(lambda x: text_to_indices(x))

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        return torch.LongTensor(self.texts[idx]), torch.FloatTensor([self.labels[idx]])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df['indices'].tolist(), df['label'].tolist(), test_size=0.2, random_state=42, stratify=df['label'])

train_dataset = NewsDataset(X_train, y_train)
test_dataset = NewsDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64)

In [ ]:
class SimpleRNNClassifier(nn.Module):
    def __init__(self, vocab_size=len(vocab), emb_dim=100, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.rnn = nn.RNN(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
    
    def forward(self, x):
        emb = self.embedding(x)
        _, hn = self.rnn(emb)
        out = self.fc(hn[-1]).squeeze()
        return torch.sigmoid(out), out

In [ ]:
class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size=len(vocab), emb_dim=100, hidden_dim=128, bidirectional=True):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden_dim, batch_first=True, bidirectional=bidirectional)
        factor = 2 if bidirectional else 1
        self.fc = nn.Linear(hidden_dim * factor, 1)
    
    def forward(self, x):
        emb = self.embedding(x)
        _, (hn, _) = self.lstm(emb)
        last = hn[-1] if not self.lstm.bidirectional else torch.cat((hn[-2], hn[-1]), dim=1)
        out = self.fc(last).squeeze()
        return torch.sigmoid(out), out

In [ ]:
def train_model(model, train_loader, test_loader, epochs=10):
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    history = []
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for texts, labels in train_loader:
            optimizer.zero_grad()
            _, logits = model(texts)
            loss = criterion(logits, labels.squeeze())
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            total_loss += loss.item()
        
        model.eval()
        preds, trues = [], []
        with torch.no_grad():
            for texts, labels in test_loader:
                _, logits = model(texts)
                preds.extend(torch.sigmoid(logits) > 0.5)
                trues.extend(labels.squeeze())
        
        acc = accuracy_score(trues, preds)
        f1 = f1_score(trues, preds)
        print(f"Epoch {epoch+1}: Loss = {total_loss/len(train_loader):.4f}, Test Acc = {acc:.4f}, F1 = {f1:.4f}")
        history.append((acc, f1))
    
    return model, history

In [ ]:
rnn_model = SimpleRNNClassifier()
lstm_model = LSTMClassifier(bidirectional=True)

print("Навчаємо RNN...")
rnn_model, rnn_hist = train_model(rnn_model, train_loader, test_loader, epochs=8)

print("\nНавчаємо LSTM...")
lstm_model, lstm_hist = train_model(lstm_model, train_loader, test_loader, epochs=8)



In [ ]:
def evaluate(model, loader):
    model.eval()
    preds, trues, probs = [], [], []
    with torch.no_grad():
        for texts, labels in loader:
            prob, logits = model(texts)
            preds.extend((prob > 0.5).int().tolist())
            trues.extend(labels.int().tolist())
            probs.extend(prob.tolist())
    print(classification_report(trues, preds))
    print("ROC AUC:", roc_auc_score(trues, probs))

print("RNN Results:")
evaluate(rnn_model, test_loader)
print("\nLSTM Results:")
evaluate(lstm_model, test_loader)